In [ ]:
import os
import sys
dir_path = os.getcwd()
print("The directory of this script is:", dir_path)
root_path = os.path.dirname(dir_path)
sys.path.append(root_path)
print("The root directory is:", root_path)

In [ ]:
import pandas as pd
import numpy as np
data = pd.read_csv(f"{root_path}\data\Telco-Customer-Churn.csv")
data["TotalCharges"] = pd.to_numeric(data["TotalCharges"], errors="coerce").fillna(data["MonthlyCharges"]*(data["tenure"])).astype(float)
data['gender'] = data['gender'].map({'Male': 1, 'Female': 0})
data['Contract'] = data['Contract'].map({'Month-to-month': 0, 'One year': 1, 'Two year': 2})
for col in data.columns:
    if col == 'Churn':
        continue
    unique_vals = data[col].dropna().unique()
    if set(unique_vals) <= {'Yes', 'No'}:
        data[col] = data[col].map({'No': 0, 'Yes': 1})

for col in data.columns:
    if col == 'Churn':
        continue
    unique_vals = data[col].dropna().unique()
    if 'No phone service' in unique_vals or 'No internet service' in unique_vals:
        mapping = {'No phone service': -1, 'No internet service': -1, 'No': 0, 'Yes': 1}
        data[col] = data[col].map(mapping)

data

In [ ]:
edges = {}
bin_cols = ["tenure", "MonthlyCharges", "TotalCharges"]
edges = {
    "tenure": [0, 2, 6, 12, 19, 29, 40, 50, 61, 69, float("inf")],
    "MonthlyCharges" : [0, 20.05, 25.05, 45.40, 59.00, 70.55, 79.25, 85.74, 94.55, 103.28, float("inf")],
    "TotalCharges": [0, 84.44, 269.81, 543.95, 928.88, 1372.45, 2071.61, 3248.81, 4509.76, 5969.91, float("inf")]
}

for col in bin_cols:
    data[f"{col}_bin"] = pd.cut(
        data[col],
        bins=edges[col],
        labels=False,
        include_lowest=True
    )
columns_to_drop = ['tenure', 'MonthlyCharges', 'TotalCharges']
numrical_columns = data[columns_to_drop]
data = data.drop(columns_to_drop, axis=1)

id_column = data['customerID']
churn_column = data['Churn']
data = data.drop(["Churn","customerID"], axis=1)

In [ ]:
import pickle
with open("output/churn_tree_model.pkl", "rb") as f:
    clf = pickle.load(f)

In [ ]:
BETA = 3
probs_uncalibrated = clf.predict_proba(data)[:, 1]
probs_calibrated = ((1/BETA) * probs_uncalibrated) / ((1/BETA) * probs_uncalibrated + (1 - probs_uncalibrated))

data['Churn_Probability'] = probs_calibrated*100


In [ ]:
from sklearn.metrics import roc_curve, roc_auc_score
probs = clf.predict_proba(X_val)[:, 1]
probs_cal = ((1/BETA) * probs) / ((1/BETA) * probs + (1 - probs))

fpr, tpr, thresholds = roc_curve(y_val.map({"No":0, "Yes":1}), probs_cal)
auc = roc_auc_score(y_val.map({"No":0, "Yes":1}), probs_cal)

plt.figure(figsize=(6,5))
plt.plot(fpr, tpr, label=f"AUC = {auc:.4f}")
plt.plot([0, 1], [0, 1], linestyle="--")
plt.xlabel("False Positive Rate")
plt.ylabel("True Positive Rate (Recall)")
plt.title("ROC Curve")
plt.legend()
plt.grid(True)
plt.show()


In [ ]:
from sklearn.metrics import precision_recall_curve, average_precision_score
probs = clf.predict_proba(X_val)[:, 1]
probs_cal = ((1/BETA) * probs) / ((1/BETA) * probs + (1 - probs))
precision, recall, thresholds = precision_recall_curve(y_val, probs_cal, pos_label="Yes")
ap = average_precision_score(y_val, probs_cal, pos_label="Yes")

plt.figure(figsize=(6, 5))
plt.plot(recall, precision, linewidth=2)
plt.xlabel("Recall")
plt.ylabel("Precision")
plt.title(f"Precision–Recall Curve (AP = {ap:.3f})")
plt.grid(True)
plt.show()


In [ ]:
best_t = None
best_score = -1
probs = clf.predict_proba(X_val)[:, 1]
probs_cal = ((1/BETA) * probs) / ((1/BETA) * probs + (1 - probs))
for t in thresholds:
    preds = (probs_cal >= t).astype(int)
    preds = np.where(preds == 1, "Yes", "No")
    score = fbeta_score(y_val, preds,pos_label="Yes", beta=2)
    if score > best_score:
        best_score = score
        best_t = t
print("Best threshold for F2 score:", round(best_t,4))
print("Best F2 score:", round(best_score,4))

In [ ]:
for name, features_set, labels_set in [('Train', X_train, y_train), ('Validation', X_val, y_val)]:
    probs = clf.predict_proba(features_set)[:, 1]
    probs_cal = ((1/BETA) * probs) / ((1/BETA) * probs + (1 - probs))
    y_predicted = (probs_cal >= best_t).astype(int)
    y_predicted = np.where(y_predicted == 1, "Yes", "No")
    acc = accuracy_score(labels_set, y_predicted)
    prec = precision_score(labels_set, y_predicted, pos_label="Yes")
    rec = recall_score(labels_set, y_predicted, pos_label="Yes")
    fbeta = fbeta_score(labels_set, y_predicted, beta=2, pos_label="Yes")
    print(f"{name} - Accuracy: {acc:.4f} - Precision: {prec:.4f} - Recall: {rec:.4f} - F2: {fbeta:.4f}")

In [ ]:
idx = np.argmin(np.abs(thresholds - best_t))
best_p = precision[idx]
best_r = recall[idx]

plt.figure(figsize=(6, 5))
plt.plot(recall, precision, linewidth=2)

# Mark the chosen threshold
plt.scatter(best_r, best_p, color="red", s=60)
plt.text(best_r, best_p,
         f"  τ={best_t:.2f}\n  P={best_p:.2f}, R={best_r:.2f}",
         verticalalignment="bottom")

plt.xlabel("Recall")
plt.ylabel("Precision")
plt.title(f"Precision–Recall Curve (AP = {ap:.3f})")
plt.grid(True)
plt.show()

In [ ]:
plt.figure(figsize=(200,40))
plot_tree(
    clf,
    feature_names=X.columns,
    class_names=[str(c) for c in clf.classes_],
    filled=True,
    rounded=True,
    fontsize=12
)
plt.savefig("output/churn_tree_plot.png")
plt.show()

In [ ]:
#final testing
probs = clf.predict_proba(X_test)[:, 1]
probs_cal = ((1/BETA) * probs) / ((1/BETA) * probs + (1 - probs))
y_predicted = (probs_cal >= best_t).astype(int)
y_predicted = np.where(y_predicted == 1, "Yes", "No")
acc = accuracy_score(y_test, y_predicted)
prec = precision_score(y_test, y_predicted, pos_label="Yes")
rec = recall_score(y_test, y_predicted, pos_label="Yes")
fbeta = fbeta_score(y_test, y_predicted, beta=2, pos_label="Yes")
print(f"Test - Accuracy: {acc:.4f} - Precision: {prec:.4f} - Recall: {rec:.4f} - F2: {fbeta:.4f}")


In [ ]:
fpr, tpr, thresholds = roc_curve(y_test.map({"No":0, "Yes":1}), probs_cal)
auc = roc_auc_score(y_test.map({"No":0, "Yes":1}), probs_cal)

plt.figure(figsize=(6,5))
plt.plot(fpr, tpr, label=f"AUC = {auc:.4f}")
plt.plot([0, 1], [0, 1], linestyle="--")
plt.xlabel("False Positive Rate")
plt.ylabel("True Positive Rate (Recall)")
plt.title("ROC Curve")
plt.legend()
plt.grid(True)
plt.show()

In [ ]:
precision, recall, thresholds = precision_recall_curve(y_test, probs_cal, pos_label="Yes")
ap = average_precision_score(y_test, probs_cal, pos_label="Yes")
idx = np.argmin(np.abs(thresholds - best_t))
best_p = precision[idx]
best_r = recall[idx]

plt.figure(figsize=(6, 5))
plt.plot(recall, precision, linewidth=2)

plt.scatter(best_r, best_p, color="red", s=60)
plt.text(best_r, best_p,
         f"  τ={best_t:.2f}\n  P={best_p:.2f}, R={best_r:.2f}",
         verticalalignment="bottom")

plt.xlabel("Recall")
plt.ylabel("Precision")
plt.title(f"Precision–Recall Curve (AP = {ap:.3f})")
plt.grid(True)
plt.show()

In [ ]:
plt.figure(figsize=(8, 8))
ax = plt.gca()

# Perfect calibration reference line
ax.plot([0, 1], [0, 1], "k:", label="Perfect Calibration")

# Uncalibrated curve (red)
frac_pos_uncal, mean_pred_uncal = calibration_curve(y_test, probs, n_bins=20, pos_label="Yes")
ax.plot(
    mean_pred_uncal,
    frac_pos_uncal,
    "s-",
    label="Uncalibrated",
    color="red"
)
# Calibrated curve (blue)
frac_pos_cal, mean_pred_cal = calibration_curve(y_test, probs_cal, n_bins=20, pos_label="Yes")
ax.plot(
    mean_pred_cal,
    frac_pos_cal,
    "s-",
    label="Custom Calibrated",
    color="blue"
)

# Formatting
ax.set_xlabel("Mean Predicted Probability")
ax.set_ylabel("Fraction of Positives")
ax.set_title("Reliability Diagram")
ax.legend(loc="lower right")
ax.grid(True, linestyle="--", alpha=0.6)

plt.show()


In [ ]:
import pickle
with open("output/churn_tree_model.pkl", "wb") as f:
    pickle.dump(clf, f)

In [ ]:
import shap
explainer = shap.TreeExplainer(clf)
shap_vals = explainer.shap_values(X_train)
print(shap_vals.shape)
shap_use = shap_vals[:, :, 1]

mean_abs = np.abs(shap_use).mean(axis=0)
df_shap = pd.Series(mean_abs, index=X_train.columns).sort_values(ascending=False)
print(df_shap)

In [ ]:
shap.summary_plot(shap_use, X_train, plot_size=[12,8])

In [ ]:
plt.figure(figsize=(16, 8))
shap_plot_bar = shap.summary_plot(shap_use, X_train, plot_type="bar", plot_size=[12,6])

In [ ]:
from sklearn.metrics import make_scorer
from sklearn.inspection import permutation_importance

f2_scorer = make_scorer(fbeta_score, beta=2, pos_label="Yes")

result = permutation_importance(
    clf,
    X_test,
    y_test,
    n_repeats=1000,
    random_state=42,
    scoring=f2_scorer
)

perm_importances = pd.Series(result.importances_mean, index=X_test.columns)
perm_importances = perm_importances.sort_values(ascending=False)
print(perm_importances)


In [ ]:
plt.figure(figsize=(8,10))
perm_importances.plot(kind='barh')
plt.title("Permutation Feature Importances")
plt.xlabel("Importance (mean decrease in score)")
plt.ylabel("Features")
plt.gca().invert_yaxis()
plt.axvline(0, color='lime', linestyle='-')
plt.show()
